# RQ1 reproducibility notebook

This notebook re-runs the RQ1 analysis from our CCS 2026 submission end to
end. It is organised as a funnel: each cell starts from the population the
previous cell ended with, adds one more condition, and prints how many first parties
or third parties survived.

Every number is printed next to the value quoted in the paper, so the
comparison is in the output itself — you do not need to open the paper to
follow along. Run all cells top to bottom (Cell → Run All); end-to-end
runtime is about a minute on a laptop once the archive is extracted.


## 1. Setup


### 1.1 Decompress the bundled dataset

`data/dataset.tar.gz` holds the HPC crawl outputs used by the paper.
This cell unpacks it into `data/raw/`. Safe to re-run — it skips extraction
if the files are already there.


In [ ]:
import os, tarfile, pathlib

REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebook' else pathlib.Path.cwd()
DATA_DIR  = REPO_ROOT / 'data' / 'raw'
BUNDLE    = REPO_ROOT / 'data' / 'dataset.tar.gz'

DATA_DIR.mkdir(parents=True, exist_ok=True)
if (DATA_DIR / 'results.jsonl').exists():
    print(f'Already extracted at {DATA_DIR}')
else:
    assert BUNDLE.exists(), f'Missing {BUNDLE}. Did you clone with the tarball?'
    print(f'Extracting {BUNDLE.name} ...')
    with tarfile.open(BUNDLE, 'r:gz') as tf:
        tf.extractall(DATA_DIR)
    print('Done.')

total_mb = sum(p.stat().st_size for p in DATA_DIR.iterdir()) / 1e6
print(f'\n{len(list(DATA_DIR.iterdir()))} files, {total_mb:.0f} MB extracted.')


### 1.2 Imports and shared helpers


In [ ]:
import json, re, glob, collections
import numpy as np
import matplotlib.pyplot as plt

MIN_WORDS = 500
WORD_RE   = re.compile(r'\S+')

def wc_en(entry):
    '''English word count of a TP cache entry; None if non-English or missing.'''
    if not isinstance(entry, dict):
        return None
    if entry.get('language', 'en') != 'en':
        return None
    wc = entry.get('word_count')
    if wc is None:
        wc = len(WORD_RE.findall(entry.get('text') or ''))
    return wc

def pct(xs, p):
    xs = sorted(xs); k = (len(xs) - 1) * p / 100
    f = int(k); c = min(f + 1, len(xs) - 1)
    return xs[f] + (xs[c] - xs[f]) * (k - f)

def line(label, count, previous=None, ref=None):
    '''Pretty-print a funnel row: count, drop from previous, paper comparison.'''
    s = f'  {label:<48s} {count:>7,}'
    if previous is not None and previous > 0:
        s += f'   ({100*count/previous:5.1f}% of prev step)'
    if ref is not None:
        s += f'   [paper: {ref}]'
    print(s)


### 1.3 Load the dataset into memory

One JSON row per crawled first party (`results.jsonl`), plus the third-party policy
cache (shards + LLM rediscovery back-patch) and the hand-curated blacklists.


In [ ]:
rows = [json.loads(ln) for ln in open(DATA_DIR / 'results.jsonl')]
print(f'rows in results.jsonl                : {len(rows):>7,}')

tp_cache = {}
for f in sorted(glob.glob(str(DATA_DIR / 'results_shard*.tp_cache.json'))):
    tp_cache.update(json.load(open(f)))
print(f'entries in third-party policy cache  : {len(tp_cache):>7,}')

bl = json.load(open(DATA_DIR / 'policy_quality_blacklist.json'))
fp_blacklist       = set(bl.get('fp_blacklist_etld1', []))
tp_blacklist_urls  = set(bl.get('tp_blacklist_urls', []))
print(f'blacklisted FP eTLD+1s               : {len(fp_blacklist):>7,}')
print(f'blacklisted TP policy URLs           : {len(tp_blacklist_urls):>7,}')

# Rediscovery back-patch: TP eTLD+1 → discovered policy URL.
redisc = {e['rediscovered_from_etld1'].lower(): u
          for u, e in tp_cache.items()
          if isinstance(e, dict) and e.get('rediscovered_from_etld1')}
print(f'policy URLs recovered by rediscovery : {len(redisc):>7,}')


## 2. First-party funnel

Take every first party we attempted to crawl and see how many survive
each filter on the way to "has a qualifying privacy policy". Reproduces the
top block of `tab:pipeline-attrition` and the first-party column of
`tab:rq1`.


### 2.1 From the Tranco seed to a crawled first party

Shows the top of `tab:pipeline-attrition`: we start from a 16,100-rank
Tranco slice, intersect with Chrome UX Report + an HTTP pre-flight probe,
then attempt a full crawl on every survivor.


In [ ]:
summary = json.load(open(DATA_DIR / 'results.summary.json'))
status_counts = summary.get('status_counts', {})

TRANCO_SEED = 16_100
AFTER_CRUX  = 7_489   # from the paper's pipeline section
processed   = summary['processed_sites']
home_ok     = summary['home_ok_count']
policy_url  = summary['policy_found_count']   # status == 'ok'
no_policy   = status_counts.get('policy_not_found', 0)

print('Funnel 2.1 — first party funnel')
line('Tranco top-16,100 seed',              TRANCO_SEED)
line('After CrUX + HTTP pre-flight',        AFTER_CRUX,  TRANCO_SEED, '7,489')
line('Processed (full crawl attempt)',      processed,   AFTER_CRUX,  '7,488')
line('  Home rendered ok',                  home_ok,     AFTER_CRUX,  '5,396 (72.1%)')
line('    Policy URL discovered',           policy_url,  home_ok,     '4,535 (60.6%)')
line('    Policy not discovered',           no_policy,   home_ok,       '861 (11.5%)')
line('  Non-browsable',                     status_counts.get('non_browsable', 0),    AFTER_CRUX, '1,067 (14.2%)')
line('  Home-fetch failed',                 status_counts.get('home_fetch_failed', 0), AFTER_CRUX,  '981 (13.1%)')
line('  Rendering exception',               status_counts.get('exception', 0),         AFTER_CRUX,    '44 (0.6%)')


### 2.2 From a discovered policy to a qualifying one

The paper counts a policy as *qualifying* only if it is (a) in English and
(b) at least 500 words long, and the first party is not on our FP quality
blacklist. This cell drops the 4,535 policy-URL-known first parties one
condition at a time.


In [ ]:
step_url       = sum(1 for r in rows if r.get('status') == 'ok')
step_english   = sum(1 for r in rows if r.get('status') == 'ok' and r.get('policy_is_english'))
step_500w      = sum(1 for r in rows if r.get('status') == 'ok' and r.get('policy_is_english')
                     and (r.get('first_party_policy_word_count') or 0) >= MIN_WORDS)
step_notblack  = sum(1 for r in rows if r.get('status') == 'ok' and r.get('policy_is_english')
                     and (r.get('first_party_policy_word_count') or 0) >= MIN_WORDS
                     and (r.get('site_etld1') or '').lower() not in fp_blacklist)

print('Funnel 2.2 — first-party policy qualification')
line('Policy URL known (status = ok)',        step_url)
line('+ English',                              step_english,  step_url)
line('+ at least 500 words',                   step_500w,     step_english)
line('+ not on FP quality blacklist',          step_notblack, step_500w,  '3,067 (qualifying FP)')


### 2.3 First-party availability (Table 1, first-party column)


In [ ]:
fp_observed   = len(rows)
fp_url_known  = step_url
fp_qualifying = step_notblack

print('Table 1 — first-party column')
print(f'  Observed domains        {fp_observed:>6,}   [paper 7,488]')
print(f'  Policy URL known        {fp_url_known:>6,}   [paper 4,535]')
print(f'  Qualifying policy       {fp_qualifying:>6,}   [paper 3,067]')
print(f'  Availability            {100*fp_qualifying/fp_observed:5.1f}%   [paper 41.0%]')
print(f'  Retention (qual / URL)  {100*fp_qualifying/fp_url_known:5.1f}%   [paper 67.6%]')


## 3. Third-party funnel

Same idea, but now for the third-party domains that showed up on those
crawled first parties. Reproduces the third-party column of `tab:rq1`.


### 3.1 From every observed third-party domain to a qualifying policy

We look at every `third_parties[*]` entry across all successfully-rendered
first-party pages, deduplicate by eTLD+1, and walk down the same
English/≥500w/not-blacklisted funnel.


In [ ]:
tp_observed       = set()
tp_url_known      = set()
tp_url_english    = set()
tp_url_500w       = set()   # == qualifying
tp_url_to_wc      = {}      # policy URL → word count (for length stats later)

for r in rows:
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict):
            continue
        tet = (tp.get('third_party_etld1') or '').lower()
        if not tet:
            continue
        tp_observed.add(tet)
        pu = tp.get('policy_url') or redisc.get(tet)
        if pu and pu in tp_blacklist_urls:
            pu = None
        if not pu:
            continue
        tp_url_known.add(tet)
        entry = tp_cache.get(pu)
        if not isinstance(entry, dict): continue
        if entry.get('language', 'en') != 'en': continue
        tp_url_english.add(tet)
        w = entry.get('word_count') or len(WORD_RE.findall(entry.get('text') or ''))
        if w >= MIN_WORDS:
            tp_url_500w.add(tet)
            tp_url_to_wc[pu] = w

print('Funnel 3.1 — third-party policy qualification')
line('Third-party eTLD+1s observed',          len(tp_observed))
line('+ policy URL known (direct or redisc)', len(tp_url_known),  len(tp_observed))
line('+ English',                              len(tp_url_english), len(tp_url_known))
line('+ at least 500 words',                   len(tp_url_500w),   len(tp_url_english), '1,122 (qualifying TP)')
print()
print('Table 1 — third-party column')
print(f'  Observed domains        {len(tp_observed):>6,}   [paper 4,771]')
print(f'  Policy URL known        {len(tp_url_known):>6,}   [paper 1,334]')
print(f'  Qualifying policy       {len(tp_url_500w):>6,}   [paper 1,122]')
print(f'  Availability            {100*len(tp_url_500w)/len(tp_observed):5.1f}%   [paper 23.5%]')
print(f'  Retention (qual / URL)  {100*len(tp_url_500w)/len(tp_url_known):5.1f}%   [paper 84.1%]')


### 3.2 Mapping third parties to canonical entities

Two different definitions of "mapped" float around in the data:

1. **Tagged** — the domain was matched against a Tracker Radar or
   TrackerDB pattern file (one of the `*_source_*_file` fields is set).
2. **Resolved to a canonical entity** — in addition, the `entity` field is
   filled in (e.g. `"Google LLC"`, `"Criteo SA"`).

Definition 1 includes domains that Tracker Radar/TrackerDB have heard of
but do not yet place under a named owner. The paper uses definition 2
(39.7% of qualified-corpus TPs), because the cross-policy analysis works
at the entity level. This cell shows both so the difference is visible.


In [ ]:
def index_status(tp):
    '''Returns a 2-tuple: (in_index, has_entity).'''
    in_index = bool(tp.get('tracker_radar_source_domain_file')
                    or tp.get('trackerdb_source_pattern_file')
                    or tp.get('trackerdb_source_org_file'))
    return in_index, bool(tp.get('entity'))

tagged_any = set()
resolved_any = set()
for r in rows:
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict): continue
        tet = (tp.get('third_party_etld1') or '').lower()
        if not tet: continue
        in_idx, has_ent = index_status(tp)
        if in_idx:           tagged_any.add(tet)
        if in_idx and has_ent: resolved_any.add(tet)

print('Indexing status, across all observed third parties')
line('Observed TP eTLD+1s',                       len(tp_observed))
line('Tagged in Tracker Radar / TrackerDB',       len(tagged_any),   len(tp_observed))
line('  of which: resolved to a canonical entity', len(resolved_any), len(tagged_any))
line('  of which: tagged but NO entity',           len(tagged_any) - len(resolved_any), len(tagged_any))


## 4. Cross-policy corpus

This is the strictest filter: we keep a first party only if it has a qualifying
first-party policy **and** at least one embedded third party that *also*
has a qualifying policy. Both sides need a policy, otherwise there is no
pair for the cross-policy analysis. This reproduces the second block of
`tab:pipeline-attrition`.


### 4.1 Qualified first parties (both sides qualify)


In [ ]:
qualified_sites = set()
tp_entity_map   = {}   # tet → entity (used by the same-org filter below)

for r in rows:
    site_et = (r.get('site_etld1') or '').lower()
    fp_q = (r.get('status') == 'ok'
            and r.get('policy_is_english')
            and (r.get('first_party_policy_word_count') or 0) >= MIN_WORDS
            and site_et not in fp_blacklist)
    if not fp_q:
        continue
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict): continue
        tet = (tp.get('third_party_etld1') or '').lower()
        if not tet: continue
        if tp.get('entity'):
            tp_entity_map[tet] = tp['entity']
        pu = tp.get('policy_url') or redisc.get(tet)
        if pu and pu in tp_blacklist_urls: pu = None
        if not pu: continue
        w = wc_en(tp_cache.get(pu))
        if w is not None and w >= MIN_WORDS:
            qualified_sites.add(site_et)
            break

print('Funnel 4.1 — qualified first parties')
line('Qualifying FP (from §2.2)',              fp_qualifying)
line('+ has ≥1 qualifying TP on the page',     len(qualified_sites), fp_qualifying, '2,755')


### 4.2 Same-organisation exclusion

Drop first parties where every embedded third party resolves to the *same* entity
as the first party — e.g. `youtube.com` embedded inside `google.com`. Our
heuristic differs slightly from the paper's canonical filter (which uses a
richer entity-group mapping); the delta is a handful of first parties, and the
final pair count drifts by about 0.2%.


In [ ]:
same_org_dropped = set()
for r in rows:
    site_et = (r.get('site_etld1') or '').lower()
    if site_et not in qualified_sites: continue
    fp_ent = tp_entity_map.get(site_et)
    if not fp_ent: continue
    tp_ents = [tp.get('entity') for tp in (r.get('third_parties') or [])
               if isinstance(tp, dict) and tp.get('entity')]
    if tp_ents and all(e == fp_ent for e in tp_ents):
        same_org_dropped.add(site_et)

print('Funnel 4.2 — same-organisation filter')
line('Qualified first parties',                        len(qualified_sites))
line('− same-org-only (heuristic)',            len(qualified_sites) - len(same_org_dropped))
print('  [paper canonical: 2,735 after same-org; drift is minor]')


### 4.3 Third parties on the qualified corpus

Now restrict attention to the third parties that appear on a qualified first party.
Paper numbers: 3,408 observed → 1,354 resolved to an entity → 996 with a
qualifying policy of their own.


In [ ]:
tps_obs_q    = set()
tps_mapped_q = set()   # index+entity (paper's "mapped" definition)
tps_qual_q   = set()

for r in rows:
    site_et = (r.get('site_etld1') or '').lower()
    if site_et not in qualified_sites: continue
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict): continue
        tet = (tp.get('third_party_etld1') or '').lower()
        if not tet: continue
        tps_obs_q.add(tet)
        in_idx, has_ent = index_status(tp)
        if in_idx and has_ent:
            tps_mapped_q.add(tet)
        pu = tp.get('policy_url') or redisc.get(tet)
        if pu and pu in tp_blacklist_urls: pu = None
        if pu:
            w = wc_en(tp_cache.get(pu))
            if w is not None and w >= MIN_WORDS:
                tps_qual_q.add(tet)

n = len(tps_obs_q)
print('Table 2 — third parties on qualified first parties')
print(f'  Observed                            {n:>5,}   [paper 3,408]')
print(f'  Resolved to a canonical entity      {len(tps_mapped_q):>5,}   [paper 1,354  (39.7%)]')
print(f'  With a qualifying English policy    {len(tps_qual_q):>5,}   [paper   996  (29.2%)]')


### 4.4 Cross-policy (first party, third party) pairs

One pair per (qualified first party, qualifying TP on that first party), minus pairs where
both sides are the same entity. Paper's canonical count is 30,956; our
simpler same-org heuristic lands ~0.2% higher.


In [ ]:
pairs = []
for r in rows:
    site_et = (r.get('site_etld1') or '').lower()
    if site_et not in qualified_sites: continue
    fp_ent = tp_entity_map.get(site_et)
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict): continue
        tet = (tp.get('third_party_etld1') or '').lower()
        if not tet: continue
        pu = tp.get('policy_url') or redisc.get(tet)
        if pu and pu in tp_blacklist_urls: pu = None
        if not pu: continue
        w = wc_en(tp_cache.get(pu))
        if w is None or w < MIN_WORDS: continue
        if fp_ent and tp.get('entity') == fp_ent:
            continue
        pairs.append((site_et, tet, pu))

print(f'Cross-policy pairs (heuristic filter): {len(pairs):,}   [paper canonical 30,956]')


## 5. Policy length distributions

Reproduces the "Median / Mean length (words)" rows in `tab:rq1`. First-party
stats are computed on the 3,067 qualifying FPs; third-party stats on
the 870 **unique** policy documents behind the 1,122 qualifying TP domains
(the gap comes from sibling domains sharing a single group-wide policy,
e.g. Google's `doubleclick.net` and `youtube.com` pointing at the same
page).


In [ ]:
fp_wcs = [r.get('first_party_policy_word_count') for r in rows
          if r.get('status') == 'ok'
          and r.get('policy_is_english')
          and (r.get('first_party_policy_word_count') or 0) >= MIN_WORDS
          and (r.get('site_etld1') or '').lower() not in fp_blacklist]
tp_wcs = list(tp_url_to_wc.values())

def describe(label, xs, ref):
    print(f'{label:<14s} n={len(xs):>5,}   '
          f'median={pct(xs,50):>6,.0f}   mean={sum(xs)/len(xs):>6,.0f}   '
          f'IQR={pct(xs,25):>5,.0f}-{pct(xs,75):<5,.0f}   [paper {ref}]')

describe('First-party', fp_wcs, 'median 3,947  mean 5,615  IQR 2,155-6,328')
describe('Third-party', tp_wcs, 'median 3,396  mean 4,982  IQR 2,020-5,300')
print(f'\nUnique TP policy URLs behind qualifying domains: {len(tp_url_to_wc):,}   [paper 870]')


## 6. Corpus scale and per-FP density

Reproduces the "Scale and composition" paragraph: total corpus size, FP/TP
split, and per-qualified-FP third-party density (median 8, mean 11, p90 24,
max 85 in the paper — the max drifts by 1 after the post-submission
augmentation pass).


In [ ]:
fp_words, tp_words = sum(fp_wcs), sum(tp_wcs)
print('Scale and composition')
print(f'  First-party policies total words  {fp_words:>12,}  ({fp_words/1e6:4.1f} M)   [paper 17.2 M]')
print(f'  Third-party policies total words  {tp_words:>12,}  ({tp_words/1e6:4.1f} M)   [paper 4.3 M]')
print(f'  Grand total                       {fp_words+tp_words:>12,}  ({(fp_words+tp_words)/1e6:4.1f} M)   [paper 21.5 M]')

per_site_density = collections.Counter()
for r in rows:
    site_et = (r.get('site_etld1') or '').lower()
    if site_et not in qualified_sites: continue
    seen = set()
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict): continue
        tet = (tp.get('third_party_etld1') or '').lower()
        if tet in tps_qual_q and tet not in seen:
            per_site_density[site_et] += 1
            seen.add(tet)

vals = sorted(per_site_density.values())
print(f'\nPer-qualified-first party third-party density')
print(f'  median {pct(vals,50):.0f}   mean {sum(vals)/len(vals):.1f}   p90 {pct(vals,90):.0f}   max {max(vals)}   [paper: 8 / 11 / 24 / 85]')


## 7. Per-category breakdown (two views)

The paper reports per-CrUX-category counts twice with slightly different
denominators:

- The "First party axis" paragraph cites numbers on the **qualified
  cross-policy corpus** (≈2,735 first parties), since that's the corpus the rest of
  the analysis uses.
- The category availability **figure** uses the broader qualifying-FP
  population (3,067), because availability is defined per role, not per
  pair.

This cell prints both so there's no ambiguity about which is which.


In [ ]:
per_cat_total        = collections.Counter()   # home_ok first parties per category (figure denominator)
per_cat_qual         = collections.Counter()   # all qualifying FPs (figure numerator)
per_cat_qual_corpus  = collections.Counter()   # qualifying FPs ∩ qualified corpus

for r in rows:
    et  = (r.get('site_etld1') or '').lower()
    cat = r.get('main_category') or 'Uncategorized'
    if r.get('home_ok') is True:
        per_cat_total[cat] += 1
    fp_q = (r.get('status') == 'ok'
            and r.get('policy_is_english')
            and (r.get('first_party_policy_word_count') or 0) >= MIN_WORDS
            and et not in fp_blacklist)
    if fp_q:
        per_cat_qual[cat] += 1
        if et in qualified_sites:
            per_cat_qual_corpus[cat] += 1

print('Per-category counts on the qualified cross-policy corpus (paper prose)')
print('  paper top-6: B&F 556, Tech 483, Ent 307, N&M 273, Edu 237, Ecom 232\n')
for cat, n in per_cat_qual_corpus.most_common():
    print(f'  {cat:<26s} {n:>4}')
print(f'\n  Total (qualified corpus):  {sum(per_cat_qual_corpus.values()):,}')
print(f'  Total (all qualifying FP): {sum(per_cat_qual.values()):,}  (figure denominator)')


## 8. Figures

The three RQ1 figures. Each shows the same two-colour palette and the
compact sizing used in the paper.


### 8.1 FP availability by CrUX content category

Vertical bars, sorted from highest to lowest availability.


In [ ]:
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'serif'],
    'axes.linewidth': 0.8,
    'axes.edgecolor': '#cccccc',
    'pdf.fonttype': 42,
})
PRIMARY = '#2171b5'

cats  = [c for c in per_cat_total if per_cat_total[c] >= 5]
avail = [100 * per_cat_qual.get(c, 0) / per_cat_total[c] for c in cats]
order = np.argsort(avail)[::-1]
cats_v, avail_v = [cats[i] for i in order], [avail[i] for i in order]

fig, ax = plt.subplots(figsize=(3.35, 2.6))
x = np.arange(len(cats_v))
ax.bar(x, avail_v, color=PRIMARY, edgecolor='#0b3d6b', linewidth=0.5, width=0.72)
ax.set_xticks(x)
ax.set_xticklabels(cats_v, fontsize=6.5, rotation=45, ha='right')
ax.set_ylabel('Share with qualifying policy', fontsize=7.5)
ax.set_ylim(0, 100); ax.set_yticks([0, 25, 50, 75, 100])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax.tick_params(axis='y', labelsize=7); ax.tick_params(axis='x', pad=1)
ax.grid(True, axis='y', alpha=0.25, linestyle=':', linewidth=0.5)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(pad=0.3); plt.show()


### 8.2 FP availability by Tranco rank

Stacked horizontal bars across seven log-spaced rank buckets. Shows the
head-vs-tail pattern: 92% availability in the top 100, tapering to ~48-57%
past rank 5,000.


In [ ]:
records = [(r.get('rank'),
             int(r.get('status') == 'ok' and r.get('policy_is_english')
                 and (r.get('first_party_policy_word_count') or 0) >= MIN_WORDS
                 and (r.get('site_etld1') or '').lower() not in fp_blacklist))
            for r in rows
            if r.get('home_ok') is True and isinstance(r.get('rank'), int)]
records.sort()
ranks = np.array([r for r, _ in records])
qual  = np.array([q for _, q in records])

buckets = [(1,100),(101,500),(501,1000),(1001,2500),(2501,5000),(5001,10000),(10001,int(ranks.max()))]
labels, avail_b = [], []
for lo, hi in buckets:
    mask = (ranks >= lo) & (ranks <= hi)
    if mask.sum() == 0: continue
    labels.append(f'{lo:,}–{hi:,}')
    avail_b.append(100 * qual[mask].sum() / mask.sum())

fig, ax = plt.subplots(figsize=(3.35, 2.5))
y = np.arange(len(labels))[::-1]
cov = np.array(avail_b); unc = 100 - cov
ax.barh(y, cov, height=0.7, color=PRIMARY, edgecolor='white', linewidth=0.5, label='Policy available')
ax.barh(y, unc, left=cov, height=0.7, color='#d9d9d9', edgecolor='white', linewidth=0.5, label='Policy not available')
for yi, p in zip(y, avail_b):
    if p >= 18:
        ax.text(p/2, yi, f'{p:.0f}%', ha='center', va='center', color='white', fontsize=6.5, fontweight='bold')
    else:
        ax.text(p+1.2, yi, f'{p:.0f}%', ha='left', va='center', color='#333', fontsize=6.5, fontweight='bold')
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=7)
ax.set_xlim(0, 102); ax.set_xticks([0, 25, 50, 75, 100])
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%' if v <= 100 else ''))
ax.set_ylabel('Tranco rank', fontsize=7.5)
ax.tick_params(axis='x', labelsize=7)
ax.grid(True, axis='x', alpha=0.22, linestyle=':', linewidth=0.5)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=2, fontsize=6.5, frameon=False,
          handlelength=1.2, handletextpad=0.4, columnspacing=1.0)
plt.tight_layout(pad=0.3); plt.show()


### 8.3 TP availability by prevalence rank

Ranks third parties by how many pages they appeared on, then shows the
share with a qualifying policy per prevalence bucket. Restricted to TPs
the scraper could actually reach (index-listed **or** homepage reachable
during rediscovery) — otherwise the tail is dominated by dark domains with
no chance of ever being covered.


In [ ]:
tp_home_reachable = set()
rfile = DATA_DIR / 'tp_rediscovery_full.jsonl'
if rfile.exists():
    bad = {'home_unreachable', 'fetch_error'}
    for ln in open(rfile):
        rec = json.loads(ln)
        d = (rec.get('domain') or rec.get('etld1') or '').lower()
        if d and rec.get('outcome') not in (None, *bad):
            tp_home_reachable.add(d)

qual_urls = {u for u, e in tp_cache.items()
             if (wc_en(e) or 0) >= MIN_WORDS and u not in tp_blacklist_urls}

tp_obs_counter = collections.Counter()
tp_covered, tp_in_index = set(), set()
for r in rows:
    if r.get('home_ok') is not True: continue
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict): continue
        tet = (tp.get('third_party_etld1') or '').lower()
        if not tet: continue
        if (tp.get('tracker_radar_source_domain_file')
                or tp.get('trackerdb_source_pattern_file')
                or tp.get('trackerdb_source_org_file')):
            tp_in_index.add(tet)
        pu = tp.get('policy_url') or redisc.get(tet)
        tp_obs_counter[tet] += 1
        if pu and pu in qual_urls:
            tp_covered.add(tet)

reachable = tp_in_index | tp_home_reachable | tp_covered
tp_obs_counter = collections.Counter({d: n for d, n in tp_obs_counter.items() if d in reachable})
tp_covered     = {d for d in tp_covered if d in reachable}
ranked = sorted(tp_obs_counter.items(), key=lambda x: -x[1])
n = len(ranked)
covered_flag = np.array([1 if d in tp_covered else 0 for d, _ in ranked])

pv_buckets = [(1,100),(101,250),(251,500),(501,1000),(1001,2000),(2001,n)]
labels, avail_b = [], []
for lo, hi in pv_buckets:
    if lo > n: continue
    hi = min(hi, n)
    labels.append(f'{lo:,}–{hi:,}')
    avail_b.append(100 * covered_flag[lo-1:hi].sum() / (hi - lo + 1))

fig, ax = plt.subplots(figsize=(3.35, 2.5))
y = np.arange(len(labels))[::-1]
cov = np.array(avail_b); unc = 100 - cov
ax.barh(y, cov, height=0.7, color=PRIMARY, edgecolor='white', linewidth=0.5, label='Policy available')
ax.barh(y, unc, left=cov, height=0.7, color='#d9d9d9', edgecolor='white', linewidth=0.5, label='Policy not available')
for yi, p in zip(y, avail_b):
    if p >= 18:
        ax.text(p/2, yi, f'{p:.0f}%', ha='center', va='center', color='white', fontsize=6.5, fontweight='bold')
    else:
        ax.text(p+1.2, yi, f'{p:.0f}%', ha='left', va='center', color='#333', fontsize=6.5, fontweight='bold')
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=7)
ax.set_xlim(0, 102); ax.set_xticks([0, 25, 50, 75, 100])
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%' if v <= 100 else ''))
ax.set_ylabel('TP prevalence rank', fontsize=7.5)
ax.tick_params(axis='x', labelsize=7)
ax.grid(True, axis='x', alpha=0.22, linestyle=':', linewidth=0.5)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=2, fontsize=6.5, frameon=False,
          handlelength=1.2, handletextpad=0.4, columnspacing=1.0)
plt.tight_layout(pad=0.3); plt.show()
